In [ ]:
# orchestration libraries; model/training definitions remain in c_models
import os
import sys
import json
import time
import signal
import hashlib
import argparse
import shlex
import subprocess
import fcntl
from pathlib import Path
import nbformat
import pandas as pd
from IPython.display import display

In [ ]:
# ordered PDP tasks; scheduler infrastructure is separate from scientific task definitions
STAGES = ['a_import', 'b_prepare', 'c_models', 'd_phase1_train', 'e_phase1_test',
          'f_phase2_train', 'g_phase2_test', 'h_compare', 'f_compare']
PROFILES = ['smoke', 'pilot', 'full', 'benchmark', 'reconstruct', 'reconstruct-smoke', 'study-smoke', 'study-pilot', 'study-clean', 'study-full', 'study-matched', 'study-controls', 'study-controls-smoke']
EXCLUDED_HANDOFFS = {'executed.ipynb', 'execution.log'}

def atomic_json(value, path):
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, indent=2))
    os.replace(temporary, path)

def source_hash(repo):
    paths = [repo / stage / 'src' / f'{stage}.ipynb' for stage in STAGES] + [repo / 'pipeline_tools.ipynb']
    payload = {str(path.relative_to(repo)): [c.source for c in nbformat.read(path, 4).cells] for path in paths}
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()

def workspace(repo, experiment):
    if not experiment or experiment in ['.', '..'] or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789-_. ' for c in experiment):
        raise ValueError('experiment must be a simple directory name')
    return repo / '.local' / 'experiments' / experiment

def load_control(repo, experiment):
    root = workspace(repo, experiment)
    control = json.loads((root / 'control.json').read_text())
    if control['source_hash'] != source_hash(repo):
        raise ValueError('source changed since preparation; create a new experiment')
    return root, control

def stage_links(root, stage):
    index = STAGES.index(stage)
    if not index:
        return
    previous = 'e_phase1_test' if stage == 'f_compare' else STAGES[index - 1]
    for source in (root / previous / 'output').iterdir():
        if source.name in EXCLUDED_HANDOFFS:
            continue
        target = root / stage / 'input' / source.name
        target.unlink(missing_ok=True)
        target.symlink_to(Path('..') / '..' / previous / 'output' / source.name, target_is_directory=source.is_dir())
        handoff = root / stage / 'output' / source.name
        handoff.unlink(missing_ok=True)
        handoff.symlink_to(Path('..') / 'input' / source.name, target_is_directory=source.is_dir())

# scientific settings and the scheduler share the same run-matrix builder

def settings_namespace(repo):
    import torch
    namespace = {'torch': torch, 'os': os}
    for cell in nbformat.read(repo / 'a_import/src/a_import.ipynb', 4).cells:
        if 'settings' in cell.metadata.get('tags', []):
            exec(compile(cell.source, 'a_import settings', 'exec'), namespace)
    return namespace

def planned_jobs(repo, profile, device='cuda', resume=False):
    namespace = settings_namespace(repo)
    function = namespace['study_settings'] if profile in namespace['STUDY_PROFILES'] else namespace['experiment_settings']
    cfg = function(profile, device, resume)
    return cfg, pd.DataFrame(namespace['study_jobs'](cfg)).rename_axis('index').reset_index()

def experiment_plan(repo, profile, device='cuda', resume=False):
    cfg, jobs = planned_jobs(repo, profile, device, resume)
    return {'jobs': len(jobs), 'phase2': cfg['steps2'] > 0, 'task': cfg.get('task', 'transformed_images')}

def show_plan(repo, profile):
    cfg, jobs = planned_jobs(repo, profile)
    specs = pd.DataFrame(cfg['models']).rename(columns={'name': 'model'})
    frame = jobs.merge(specs, on='model', validate='many_to_one')
    columns = ['index', 'model', 'regime', 'replicate']
    columns += [key for key in ['requested_width', 'width', 'message_bits', 'kind', 'reliability', 'capacity_mode'] if key in frame]
    display(pd.DataFrame([{'profile': profile, 'task': cfg.get('task'), 'training_tasks': len(jobs), 'state_dim': cfg['state_dim'],
                           'optimizer_steps_per_task': cfg['steps1'], 'training_replicates': cfg['replicates'], 'phase2': cfg['steps2'] > 0}]))
    display(frame[columns])
    return frame[columns]

def prepare_study_evaluation(root):
    collect_phase(root, 1)
    stage_links(root, 'e_phase1_test')

def collect_study_evaluation(root):
    jobs = pd.read_csv(root / 'job_matrix.csv')
    runs = pd.read_csv(root / 'd_phase1_train/output/phase1_manifest.csv')
    rows = []
    for job in jobs.to_dict('records'):
        run = runs.loc[runs['model'].eq(job['model']) & runs['regime'].eq(job['regime']) & runs['replicate'].eq(job['replicate'])].iloc[0]
        folder = root / 'e_phase1_test/output/eval_runs' / job['run']
        result = json.loads((folder / 'completion.json').read_text())
        checkpoint = root / 'd_phase1_train/output/phase1_runs' / run['checkpoint']
        if result['run'] != job['run'] or result['index'] != job['index'] or result['training_signature'] != run['signature']:
            raise ValueError(f'evaluation identity mismatch: {job["run"]}')
        if hashlib.sha256(checkpoint.read_bytes()).hexdigest() != result['checkpoint_sha256']:
            raise ValueError(f'checkpoint changed since evaluation: {job["run"]}')
        for name, digest in result['files'].items():
            if hashlib.sha256((folder / name).read_bytes()).hexdigest() != digest:
                raise ValueError(f'incomplete or changed evaluation artifact: {folder / name}')
        rows.append({**job, 'directory': f'eval_runs/{job["run"]}', 'case_rows': result['case_rows'],
                     'checkpoint_sha256': result['checkpoint_sha256'], 'training_signature': result['training_signature']})
    path = root / 'e_phase1_test/output/evaluation_manifest.csv'
    pd.DataFrame(rows).to_csv(path.with_suffix('.csv.tmp'), index=False)
    path.with_suffix('.csv.tmp').replace(path)

def finish_study(root, control):
    collect_study_evaluation(root)
    stage_links(root, 'f_compare')
    execute_stage(root, control, 'f_compare')


In [ ]:
# notebook executor; captured rich output goes to the executed notebook, progress records bypass capture
EXECUTOR_SOURCE = r'''
import sys
import traceback
from pathlib import Path
import nbformat
from IPython.terminal.interactiveshell import TerminalInteractiveShell
from IPython.utils.capture import capture_output
source, destination = map(Path, sys.argv[1:3])
notebook = nbformat.read(source, 4)
shell = TerminalInteractiveShell.instance()
shell.run_line_magic('matplotlib', 'inline')
shell.display_formatter.active_types = ['text/plain', 'text/html', 'image/png', 'image/svg+xml']
for code in notebook.cells:
    with capture_output(stdout=True, stderr=True, display=True) as captured:
        result = shell.run_cell(code.source, store_history=True)
    code.execution_count = result.execution_count
    code.outputs = []
    for name in ['stdout', 'stderr']:
        if getattr(captured, name):
            code.outputs.append(nbformat.v4.new_output('stream', name=name, text=getattr(captured, name)))
    code.outputs.extend(nbformat.v4.new_output('display_data', data=item.data, metadata=item.metadata or {}) for item in captured.outputs)
    error = result.error_before_exec or result.error_in_exec
    if error:
        code.outputs.append(nbformat.v4.new_output('error', ename=type(error).__name__, evalue=str(error),
                                                  traceback=traceback.format_exception(type(error), error, error.__traceback__)))
    temporary = destination.with_name(destination.name + '.tmp')
    nbformat.write(notebook, temporary)
    temporary.replace(destination)
    if error:
        raise RuntimeError(f'{source.name}: {error}')
'''

In [ ]:
# live log streaming and child-process cleanup; terminal failures propagate to Slurm
def execute_stage(root, control, stage, index=None, warm_start=None):
    env = {**os.environ, 'NCA_PROJECT_ROOT': str(root), 'NCA_PROFILE': control['profile'],
           'NCA_DEVICE': control['device'], 'NCA_RESUME': str(int(control.get('resume', False))),
           'NCA_ALLOW_DEVICE_CHANGE': str(int(control.get('allow_device_change', False))),
           'CUBLAS_WORKSPACE_CONFIG': ':4096:8', 'NCA_RUN_INDEX': '' if index is None else str(index),
           'NCA_WARM_START': '' if warm_start is None else str(warm_start), 'PYTHONUNBUFFERED': '1'}
    output = root / stage / 'output'
    if index is not None:
        jobs = pd.read_csv(root / 'job_matrix.csv')
        job = jobs.loc[jobs['index'].eq(index)].iloc[0]
        phase = 1 if stage == 'd_phase1_train' else 2
        output = output / ('eval_runs' if stage == 'e_phase1_test' else f'phase{phase}_runs') / job['run']
        output.mkdir(parents=True, exist_ok=True)
    source = root / stage / 'src' / f'{stage}.ipynb'
    log_path = output / 'execution.log'
    process = subprocess.Popen([sys.executable, '-u', '-c', EXECUTOR_SOURCE, str(source), str(output / 'executed.ipynb')],
                               cwd=root, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, start_new_session=True)
    def terminate(signum, frame):
        raise KeyboardInterrupt(f'signal {signum}')
    previous_handler = signal.signal(signal.SIGTERM, terminate)
    try:
        with log_path.open('a', buffering=1) as log:
            for line in process.stdout:
                log.write(line)
                print(line, end='', flush=True)
        returncode = process.wait()
    except BaseException:
        if process.poll() is None:
            os.killpg(process.pid, signal.SIGTERM)
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
        raise
    finally:
        signal.signal(signal.SIGTERM, previous_handler)
    if returncode:
        raise RuntimeError(f'{stage} failed ({returncode}); see {log_path}')

def require_device(device):
    # explicit CUDA requests must never silently become CPU jobs
    import torch
    if device == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA requested but unavailable; request a GPU and use a CUDA-enabled PyTorch environment')
    return 'cuda' if device == 'auto' and torch.cuda.is_available() else 'cpu' if device == 'auto' else device

In [ ]:
# preparation writes a new isolated experiment; workers never rebuild shared input links
def prepare(repo, experiment, profile, device, resume=False, allow_device_change=False):
    root = workspace(repo, experiment)
    device = require_device(device)
    if root.exists():
        if not resume:
            raise FileExistsError(f'experiment already exists: {root}; use RESUME=True or a new experiment name')
        existing_root, control = load_control(repo, experiment)
        if control['profile'] != profile or control['device'] != device:
            raise ValueError('prepared profile/device differ; use a new experiment and import compatible checkpoints')
        if not (root / 'ready.json').is_file():
            raise RuntimeError('preparation is incomplete; preserve this directory and use a new experiment name')
        control.update(resume=True, allow_device_change=allow_device_change)
        atomic_json(control, root / 'control.json')
        return root, control
    root.mkdir(parents=True)
    control = {'profile': profile, 'device': device, 'resume': resume, 'allow_device_change': allow_device_change,
               'source_hash': source_hash(repo), 'created_at': pd.Timestamp.now(tz='UTC').isoformat()}
    atomic_json(control, root / 'control.json')
    for stage in STAGES:
        task = root / stage
        (task / 'input').mkdir(parents=True)
        (task / 'output').mkdir()
        (task / 'src').symlink_to(os.path.relpath(repo / stage / 'src', task), target_is_directory=True)
    for name in ['cat.png', 'smiley.png']:
        custom = repo / 'a_import' / 'input' / name
        if custom.is_file():
            (root / 'a_import' / 'input' / name).symlink_to(custom)
    for stage in STAGES[:3]:
        stage_links(root, stage)
        execute_stage(root, control, stage)
    cfg = json.loads((root / 'a_import/output/config.json').read_text())
    _, jobs = planned_jobs(repo, profile, device, resume)
    jobs.to_csv(root / 'job_matrix.csv', index=False)
    stage_links(root, 'd_phase1_train')
    atomic_json({'jobs': len(jobs), 'prepared_at': pd.Timestamp.now(tz='UTC').isoformat()}, root / 'ready.json')
    return root, control

In [ ]:
# controller-only manifest collection; array workers write separate result.json files
def collect_phase(root, phase):
    stage = 'd_phase1_train' if phase == 1 else 'f_phase2_train'
    cfg = json.loads((root / stage / 'input/config.json').read_text())
    namespace = {}
    for code in nbformat.read(root / stage / 'input/runtime.ipynb', 4).cells:
        exec(compile(code.source, 'runtime.ipynb', 'exec'), namespace)
    data_hash = hashlib.sha256((root / stage / 'input/dataset.pt').read_bytes()).hexdigest()
    code_hash = (root / stage / 'input/runtime.sha256').read_text()
    rows = []
    for job in pd.read_csv(root / 'job_matrix.csv').to_dict('records'):
        folder = root / stage / 'output' / f'phase{phase}_runs' / job['run']
        result = json.loads((folder / 'result.json').read_text())
        spec = next(spec for spec in cfg['models'] if spec['name'] == job['model'])
        expected = namespace['run_signature'](cfg, spec, phase, job['regime'], job['replicate'], data_hash, code_hash)
        if result['signature'] != expected or result['optimizer_steps'] != cfg[f'steps{phase}']:
            raise ValueError(f'incomplete or incompatible result: {job["run"]}')
        if (result['model'], result['regime'], result['replicate'], result['phase']) != (job['model'], job['regime'], job['replicate'], phase):
            raise ValueError(f'result identity mismatch: {folder}')
        rows.append(result)
    target = root / stage / 'output' / f'phase{phase}_manifest.csv'
    pd.DataFrame(rows).to_csv(target.with_suffix('.csv.tmp'), index=False)
    target.with_suffix('.csv.tmp').replace(target)
    return pd.DataFrame(rows)

def finish_phase(root, control, phase):
    collect_phase(root, phase)
    stages = ['e_phase1_test'] if phase == 1 else ['g_phase2_test', 'h_compare']
    for stage in stages:
        stage_links(root, stage)
        execute_stage(root, control, stage)
    cfg = json.loads((root / 'a_import/output/config.json').read_text())
    if phase == 1 and cfg.get('task') == 'canonical_reconstruction':
        display(pd.read_csv(root / 'e_phase1_test/output/reconstruction_summary.csv'))
    if phase == 1 and cfg['steps2'] > 0:
        stage_links(root, 'f_phase2_train')

def run_local(repo, experiment, profile='study-smoke', device='auto', resume=False):
    root, control = prepare(repo, experiment, profile, device, resume)
    if experiment_plan(repo, profile, device)['task'] == 'communication_study':
        execute_stage(root, control, 'd_phase1_train')
        prepare_study_evaluation(root)
        execute_stage(root, control, 'e_phase1_test')
        finish_study(root, control)
        return root
    if profile == 'benchmark':
        execute_stage(root, control, 'd_phase1_train', index=0)
        return root
    phases = [(1, 'd_phase1_train')]
    if experiment_plan(repo, profile, device)['phase2']:
        phases.append((2, 'f_phase2_train'))
    for phase, stage in phases:
        execute_stage(root, control, stage)
        finish_phase(root, control, phase)
    return root

def status(repo, experiment):
    root = workspace(repo, experiment)
    rows = []
    paths = [path for phase, stage in [(1, 'd_phase1_train'), (2, 'f_phase2_train')]
             for path in (root / stage / 'output').glob(f'phase{phase}_runs/*/progress.json')]
    for path in paths:
        value = json.loads(path.read_text())
        value['run'] = path.parent.name
        value['minutes_since_save'] = round((time.time() - path.stat().st_mtime) / 60, 1)
        rows.append(value)
    columns = ['phase', 'run', 'step', 'total_steps', 'val_loss', 'device', 'updated_at', 'minutes_since_save']
    frame = pd.DataFrame(rows, columns=columns)
    display(frame.sort_values('updated_at', ascending=False))
    config_path = root / 'a_import/output/config.json'
    if config_path.is_file() and json.loads(config_path.read_text()).get('task') == 'communication_study':
        expected = len(pd.read_csv(root / 'job_matrix.csv')) if (root / 'job_matrix.csv').is_file() else 0
        evaluated = list((root / 'e_phase1_test/output/eval_runs').glob('*/completion.json'))
        display(pd.DataFrame([{'expected_runs': expected, 'training_checkpoints': len(frame),
                               'training_at_final_step': int(frame['step'].eq(frame['total_steps']).sum()),
                               'evaluation_completed': len(evaluated),
                               'comparison_ready': (root / 'f_compare/output/comparison_complete.json').is_file()}]))
    return frame

In [ ]:
# Slurm submission; arrays and phases come from the same notebook settings used in preparation
def submit_slurm(repo, experiment, profile='study-smoke', account='', max_parallel=2, walltime='24:00:00',
                 memory='32G', cpus=2, resume=False, allow_device_change=False, dry_run=False, array_chunk_size=500):
    if profile not in PROFILES or max_parallel < 1 or cpus < 1 or array_chunk_size < 1:
        raise ValueError('invalid profile or resource request')
    root = workspace(repo, experiment)
    if root.exists() and not resume:
        raise FileExistsError(f'experiment exists: {root}; use --resume or a new name')
    logs = repo / '.local' / 'slurm_logs' / experiment
    logs.mkdir(parents=True, exist_ok=True)
    launcher = repo / 'slurm' / 'run.sh'
    plan = experiment_plan(repo, profile, 'cuda', resume)
    count = plan['jobs']
    common = ['sbatch', '--parsable', '--partition=nvgpu', '--gpus=1', '--constraint=GPU_ANY',
              f'--cpus-per-task={cpus}', f'--mem={memory}', f'--time={walltime}', '--export=ALL',
              f'--chdir={repo}', f'--output={logs}/%x-%A_%a.out']
    if account:
        common.append(f'--account={account}')
    env = {**os.environ, 'NCA_PYTHON': sys.executable}
    commands, submitted = [], []
    def launch(label, action, dependency=None, array=None, index_offset=0):
        command = common + [f'--job-name=nca-{label}']
        if dependency:
            command.append('--dependency=' + dependency)
        if array:
            command.append('--array=' + array)
        command += [str(launcher), 'worker', '--experiment', experiment, '--action', action, '--profile', profile]
        if index_offset:
            command += ['--index-offset', str(index_offset)]
        if resume:
            command.append('--resume')
        if allow_device_change:
            command.append('--allow-device-change')
        commands.append(command)
        print(shlex.join(command), flush=True)
        if dry_run:
            return f'DRY_{label}'
        output = subprocess.check_output(command, cwd=repo, env=env, text=True).strip()
        job_id = output.split(';')[0]
        if not job_id.isdigit():
            raise RuntimeError(f'unexpected sbatch output: {output}')
        submitted.append({'stage': label, 'job_id': job_id})
        return job_id
    def chunks(label, action, dependency):
        last = dependency
        for start in range(0, count, array_chunk_size):
            stop = min(count - 1, start + array_chunk_size - 1)
            last = launch(f'{label}-{start // array_chunk_size}', action, 'afterok:' + last, f'0-{stop - start}%{max_parallel}', index_offset=start)
        return last
    try:
        prep = launch('prepare', 'prepare')
        if plan['task'] == 'communication_study':
            trained = chunks('phase1', 'train1', prep)
            collected = launch('collect1', 'collect1', 'afterok:' + trained)
            evaluated = chunks('eval1', 'evaluate1', collected)
            launch('compare1', 'compare1', 'afterok:' + evaluated)
        else:
            array = '0-0%1' if profile == 'benchmark' else f'0-{count - 1}%{max_parallel}'
            phase1 = launch('phase1', 'train1', 'afterok:' + prep, array)
            if profile != 'benchmark':
                finish1 = launch('test1', 'finish1', 'afterok:' + phase1)
                if plan['phase2']:
                    phase2 = launch('phase2', 'train2', 'afterok:' + finish1, array)
                    launch('test2', 'finish2', 'afterok:' + phase2)
    finally:
        record = {'experiment': experiment, 'profile': profile, 'dry_run': dry_run,
                  'submitted': submitted, 'commands': commands, 'created_at': pd.Timestamp.now(tz='UTC').isoformat()}
        atomic_json(record, logs / ('submission_dry_run.json' if dry_run else 'submission.json'))
    return record

In [ ]:
# command-line entry point; the shell launcher executes these notebook definitions
def main(arguments=None):
    parser = argparse.ArgumentParser(description='PDP NCA pipeline: local or VACC Slurm')
    parser.add_argument('command', choices=['local', 'submit', 'worker', 'status', 'warm-start', 'plan'])
    parser.add_argument('--experiment')
    parser.add_argument('--profile', choices=PROFILES, default='study-smoke')
    parser.add_argument('--device', choices=['auto', 'cpu', 'cuda'], default='auto')
    parser.add_argument('--account', default='')
    parser.add_argument('--max-parallel', type=int, default=2)
    parser.add_argument('--time', default='24:00:00')
    parser.add_argument('--array-chunk-size', type=int, default=500)
    parser.add_argument('--mem', default='32G')
    parser.add_argument('--cpus', type=int, default=2)
    parser.add_argument('--resume', action='store_true')
    parser.add_argument('--allow-device-change', action='store_true')
    parser.add_argument('--dry-run', action='store_true')
    parser.add_argument('--action', choices=['prepare', 'train1', 'finish1', 'train2', 'finish2', 'collect1', 'evaluate1', 'compare1'])
    parser.add_argument('--index', type=int)
    parser.add_argument('--index-offset', type=int, default=0)
    parser.add_argument('--phase', type=int, choices=[1, 2], default=1)
    parser.add_argument('--checkpoint', type=Path)
    args = parser.parse_args(arguments)
    if args.command != 'plan' and not args.experiment:
        parser.error('--experiment is required for this command')
    if args.profile.startswith('study-') and (args.action in ['train2', 'finish2'] or args.command == 'warm-start' and args.phase == 2):
        parser.error('study profiles are Phase 1 only')
    repo = Path.cwd().resolve()
    if args.command == 'plan':
        return show_plan(repo, args.profile)
    if args.command == 'submit':
        return submit_slurm(repo, args.experiment, args.profile, args.account, args.max_parallel,
                            args.time, args.mem, args.cpus, args.resume, args.allow_device_change, args.dry_run, args.array_chunk_size)
    if args.command == 'local':
        return run_local(repo, args.experiment, args.profile, args.device, args.resume)
    if args.command == 'status':
        return status(repo, args.experiment)
    if args.command == 'worker' and args.action == 'prepare':
        return prepare(repo, args.experiment, args.profile, 'cuda', args.resume, args.allow_device_change)
    root, control = load_control(repo, args.experiment)
    if args.profile != control['profile']:
        raise ValueError('profile differs from prepared experiment')
    if args.command == 'worker':
        require_device('cuda')
        if args.action == 'collect1':
            return prepare_study_evaluation(root)
        if args.action == 'compare1':
            return finish_study(root, control)
        if args.action in ['train1', 'train2', 'evaluate1']:
            index = args.index if args.index is not None else args.index_offset + int(os.environ['SLURM_ARRAY_TASK_ID'])
            jobs = pd.read_csv(root / 'job_matrix.csv')
            if index not in jobs['index'].tolist():
                raise ValueError('array index outside prepared job matrix')
            stage = {'train1': 'd_phase1_train', 'train2': 'f_phase2_train', 'evaluate1': 'e_phase1_test'}[args.action]
            control.update(resume=args.resume, allow_device_change=args.allow_device_change)
            return execute_stage(root, control, stage, index)
        if args.action not in ['finish1', 'finish2']:
            parser.error('worker requires --action')
        return finish_phase(root, control, 1 if args.action == 'finish1' else 2)
    # legacy imports are intentionally weights-only; old checkpoints lack portable run metadata
    if args.checkpoint is None or args.index is None:
        parser.error('warm-start requires --checkpoint and --index')
    require_device(control['device'])
    import torch
    checkpoint = torch.load(args.checkpoint, map_location='cpu', weights_only=False)
    jobs = pd.read_csv(root / 'job_matrix.csv')
    job = jobs.loc[jobs['index'].eq(args.index)].iloc[0]
    identity = (checkpoint.get('phase'), checkpoint.get('model_name'), checkpoint.get('regime'), checkpoint.get('replicate'))
    expected = (args.phase, job['model'], job['regime'], job['replicate'])
    if identity != expected:
        raise ValueError('use the matching best.pt checkpoint with phase/model/regime/replicate metadata')
    stage = 'd_phase1_train' if args.phase == 1 else 'f_phase2_train'
    folder = root / stage / 'output' / f'phase{args.phase}_runs' / job['run']
    if (folder / 'last.pt').exists():
        raise FileExistsError('destination already has training; choose a fresh experiment')
    control['resume'] = False
    return execute_stage(root, control, stage, args.index, args.checkpoint.resolve())